# Citiz — 1. Sources et données brutes (Bronze)
Version 2.0 — 20 octobre 2026. **Runtime Python Fabric**, Bronze attaché par défaut.
Installer les dépendances de `requirements_fabric.txt` dans un environnement Fabric publié.
Importer les trois notebooks; exécuter Bronze → Silver → Gold. Chaque étape publie un identifiant de run.
Les sources facultatives absentes restent inconnues; les tables d'une ancienne exécution sont ignorées.
Le catalogue national peut être long: sélectionnez les réseaux de votre étude dans les paramètres.
Les millésimes INSEE sont explicites (2021/2022/2023); leur actualisation exige de vérifier les variables.

Correctif 2.0.1: collecte des contours par département, avec essais en cas de coupure réseau.


In [ ]:
# Cellule de paramètres Fabric. Runtime Python; Bronze doit être le lakehouse par défaut.
BASE = "/lakehouse/default"
BRONZE = "bronze"
SCHEMA = "dbo"  # "" pour un lakehouse sans schémas
LOCAL_ROOT = None  # uniquement pour les tests locaux
# [] = communes de France disponibles via geo.api.gouv.fr. Restreindre pour le pilote.
DEPARTEMENTS = []  # Exemple: ["67", "68"]. Aucun département choisi à votre place.
# Aucune limite arbitraire de flux. Sélection manuelle recommandée pour un pilote.
GTFS_DATASET_IDS = []  # IDs/slug PAN; [] = tous les GTFS du catalogue
GBFS_DATASET_IDS = []
MAX_GTFS = None
MAX_GBFS = None
DOWNLOAD_GRID = True
# Données nationales lourdes: lecture CSV par morceaux; filtrage spatial après conversion.
CSV_CHUNKSIZE = 100000
# Contrôle explicite du périmètre. Les données sources ont des géographies/millésimes différents.
HISTORIQUE_URL = "https://www.insee.fr/fr/statistiques/fichier/3698339/base-pop-historiques-1876-2023.xlsx"
FILOSOFI_URL = "https://www.insee.fr/fr/statistiques/fichier/8735162/Filosofi2021_carreaux_200m_csv.zip"
LOGEMENT_URL = "https://www.insee.fr/fr/statistiques/fichier/8581474/base-cc-logement-2022_csv.zip"
FLUX_URL = ""  # vide: résolution du lien CSV sur la page FLUX_PAGE
FLUX_PAGE = 8582949  # page INSEE des flux domicile-travail
OMPHALE_URL = "https://www.insee.fr/fr/statistiques/fichier/6652134/projdepreg_2021-2070_excel.zip"
SSMSI_DATASET = "bases-statistiques-communale-departementale-et-regionale-de-la-delinquance-enregistree-par-la-police-et-la-gendarmerie-nationales"
INPUT_DIR = f"{BASE}/Files/citiz_inputs"  # fichiers métier décrits dans le guide
# Le notebook n'invente ni temps routiers, ni trafic, ni inventaire concurrentiel exhaustif.


In [ ]:
import os, re, json, unicodedata, uuid, datetime as dt
import numpy as np
import pandas as pd
import pyarrow as pa
from deltalake import DeltaTable, write_deltalake
from scipy.spatial import cKDTree
RUN_ID = str(uuid.uuid4())
STATUS = []

def nu():
    import notebookutils
    return notebookutils

def root(lakehouse):
    if LOCAL_ROOT:
        return f"{LOCAL_ROOT}/{lakehouse}", None
    info = nu().lakehouse.get(lakehouse)
    if not isinstance(info, dict):
        info = json.loads(json.dumps(info, default=lambda x: x.__dict__))
    props = info.get("properties", {})
    path = props.get("abfsPath")
    if not path:
        raise RuntimeError(f"abfsPath absent pour {lakehouse}; contrôler notebookutils.lakehouse.get")
    return path.rstrip("/"), {"bearer_token": nu().credentials.getToken("storage"), "use_fabric_endpoint":"true"}

def table_path(lakehouse, name):
    p,o = root(lakehouse)
    return f"{p}/Tables/{SCHEMA + '/' if SCHEMA else ''}{name}",o

def read_table(lakehouse, name, required=False, run_id=None):
    p,o = table_path(lakehouse,name)
    try:
        t=DeltaTable(p,storage_options=o).to_pandas()
    except Exception as e:
        if required: raise RuntimeError(f"Table obligatoire illisible: {lakehouse}.{name}") from e
        print(f"Absente/illisible: {lakehouse}.{name}: {str(e)[:120]}")
        return pd.DataFrame()
    if run_id is not None:
        if '_run_id' not in t: return pd.DataFrame()
        t=t[t['_run_id']==run_id].copy()
    return t.drop(columns=['_run_id'],errors='ignore')

def write_table(lakehouse,name,df):
    df=df.reset_index(drop=True).copy()
    df['_run_id']=RUN_ID
    # Types explicites même pour colonnes entièrement nulles / tables vides.
    a=pa.Table.from_pandas(df,preserve_index=False)
    a=pa.Table.from_arrays([a[c].cast(pa.string()) if pa.types.is_null(a[c].type) else a[c]
                            for c in a.column_names],names=a.column_names)
    p,o=table_path(lakehouse,name)
    if LOCAL_ROOT: os.makedirs(os.path.dirname(p),exist_ok=True)
    write_deltalake(p,a,mode='overwrite',schema_mode='overwrite',storage_options=o)
    STATUS.append({'table':name,'lignes':len(df),'run_id':RUN_ID})
    print(f"{lakehouse}.{name}: {len(df):,} lignes")

def nnum(s):
    return pd.to_numeric(s.astype('string').str.replace('\u202f','',regex=False)
          .str.replace(' ','',regex=False).str.replace(',','.',regex=False),errors='coerce')

def textnorm(s):
    return s.fillna('').astype(str).map(lambda v: unicodedata.normalize('NFKD',v)
                    .encode('ascii','ignore').decode().lower().strip())

def codes(s):
    s=s.astype('string').str.strip().str.upper().str.replace(r'\.0$','',regex=True)
    return s.str.zfill(5).where(s.str.fullmatch(r'\d{1,5}|2[AB]\d{3}',na=False))

def arm_to_com(s):
    s=codes(s)
    return s.mask(s.str.match(r'751\d{2}',na=False),'75056').mask(s.str.match(r'132\d{2}',na=False),'13055').mask(s.str.match(r'6938[1-9]',na=False),'69123')

def assert_unique(df,keys,name):
    if df[keys].isna().any().any() or df.duplicated(keys).any():
        raise ValueError(f"Clé nulle ou dupliquée: {name} {keys}")

def valid_points(df):
    d=df.copy()
    d['lat']=nnum(d['lat']); d['lon']=nnum(d['lon'])
    return d[d.lat.between(-90,90)&d.lon.between(-180,180)&~((d.lat==0)&(d.lon==0))].copy()

def xyz(lon,lat):
    a,b=np.radians(np.asarray(lon,dtype=float)),np.radians(np.asarray(lat,dtype=float))
    return np.column_stack([np.cos(b)*np.cos(a),np.cos(b)*np.sin(a),np.sin(b)])

def nearest(q,refs):
    if refs.empty: return np.full(len(q),-1,dtype=int),np.full(len(q),np.nan)
    d,i=cKDTree(xyz(refs.lon,refs.lat)).query(xyz(q.lon,q.lat),k=1)
    return i,2*6371008.8*np.arcsin(np.clip(d/2,0,1))

def radius_sum(q,refs,radius,value=None):
    if refs.empty: return np.zeros(len(q))
    tree=cKDTree(xyz(refs.lon,refs.lat)); chord=2*np.sin(radius/(2*6371008.8))
    result=[]
    for start in range(0,len(q),1000):
        hits=tree.query_ball_point(xyz(q.lon.iloc[start:start+1000],q.lat.iloc[start:start+1000]),chord)
        result.extend([float(refs.iloc[h][value].sum()) if value else len(h) for h in hits])
    return np.array(result)


In [ ]:
# Setup
import os, re, glob, csv, gzip, json, time, zipfile, unicodedata, datetime as dt
import numpy as np
import requests
import pandas as pd

RAW = f"{BASE}/Files/raw"
TABLES = f"{BASE}/Tables/{SCHEMA}".rstrip("/")
os.makedirs(TABLES, exist_ok=True)
TODAY = dt.date.today().isoformat()

session = requests.Session()
session.headers["User-Agent"] = "Mozilla/5.0 (fabric-hackathon-ingestion)"
manifest = []

def out_path(source, filename):
    d = f"{RAW}/{source}/{TODAY}"
    os.makedirs(d, exist_ok=True)
    return f"{d}/{filename}"

def download(url, source, filename, retries=3, skip_existing=False):
    """Stream a file into Files/raw; never raises, so one failure does not stop the run."""
    path = out_path(source, filename)
    if skip_existing and os.path.exists(path) and os.path.getsize(path) > 0:
        manifest.append({"source":source,"url":url,"path":path,"status":"cached"})
        return path
    for attempt in range(1, retries + 1):
        try:
            with session.get(url, stream=True, timeout=(15, 180)) as r:
                r.raise_for_status()
                with open(path + ".part", "wb") as f:
                    for chunk in r.iter_content(1024 * 1024):
                        f.write(chunk)
            os.replace(path + ".part", path)
            manifest.append({"source": source, "url": url, "path": path,
                             "bytes": os.path.getsize(path), "status": "ok"})
            return path
        except Exception as e:
            if attempt == retries:
                manifest.append({"source": source, "url": url, "path": path, "status": f"error: {e}"})
                print(f"FAILED {url}: {str(e)[:120]}")
            time.sleep(2 * attempt)
    return None

def safe_name(s):
    return "".join(c if c.isalnum() or c in "-_." else "_" for c in s)[:80]

def snake(c):
    c = unicodedata.normalize("NFKD", str(c)).encode("ascii", "ignore").decode()
    c = re.sub(r"[^0-9a-zA-Z]+", "_", c).strip("_").lower()
    return c or "col"

def clean_cols(df):
    cols, seen = [], {}
    for c in df.columns:
        s = snake(c); n = seen.get(s, 0); seen[s] = n + 1
        cols.append(s if n == 0 else f"{s}_{n}")
    df.columns = cols
    return df

def num(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

CODE_NAMES = {"codgeo", "dclt", "dep", "reg", "com", "arm", "epci", "code", "insee",
              "cdep", "creg", "libgeo", "l_dclt", "source_file", "scenario", "feed_id"}

def auto_numeric(df, keep=("id", "lcog", "date", "lib", "nom")):
    """Convert a text column to numbers when every non-empty value is numeric.
    Codes (leading zeros, known code columns) stay as text."""
    for c in df.columns:
        if c in CODE_NAMES or c.startswith("codgeo") or any(k in c for k in keep):
            continue
        col = df[c]
        if not (pd.api.types.is_object_dtype(col) or pd.api.types.is_string_dtype(col)):
            continue
        nn = col.dropna()
        if nn.empty or nn.astype(str).str.fullmatch(r"0\d+").any():
            continue
        s = pd.to_numeric(col, errors="coerce")
        if s.notna().sum() == col.notna().sum():
            df[c] = s
    return df

def latest_dir(source):
    dirs = sorted(d for d in glob.glob(f"{RAW}/{source}/*") if os.path.isdir(d))
    return dirs[-1] if dirs else None

def all_dirs(source):
    return sorted(d for d in glob.glob(f"{RAW}/{source}/*") if os.path.isdir(d))

def sniff_sep(raw_bytes):
    txt = raw_bytes.decode("utf-8-sig", errors="replace")
    try:
        return csv.Sniffer().sniff(txt, delimiters=",;|\t").delimiter
    except Exception:
        return ";"

def read_csv_any(source, **kw):
    """Read a CSV from a path (.csv or .csv.gz) or an open binary file, detecting the separator."""
    if isinstance(source, str):
        opener = gzip.open if source.endswith(".gz") else open
        with opener(source, "rb") as fh:
            sep = sniff_sep(fh.read(8192))
        return pd.read_csv(source, sep=sep, dtype=str, encoding="utf-8-sig",
                           on_bad_lines="error", **kw)
    head = source.read(8192); source.seek(0)
    return pd.read_csv(source, sep=sniff_sep(head), dtype=str, encoding="utf-8-sig",
                       on_bad_lines="error", **kw)

def zip_csvs(path):
    """Yield (name, DataFrame) for every CSV inside a zip."""
    with zipfile.ZipFile(path) as zf:
        for name in zf.namelist():
            if name.lower().endswith(".csv"):
                with zf.open(name) as fh:
                    yield os.path.basename(name), read_csv_any(io_bytes(fh))

def io_bytes(fh):
    import io
    return io.BytesIO(fh.read())

def old_header_row(raw):
    """First row where at least half of the widest row's cells are filled."""
    width = raw.notna().sum(axis=1).max()
    for i in range(len(raw)):
        if raw.iloc[i].notna().sum() >= max(2, width * 0.5):
            return i
    return 0

def excel_tables(path, min_rows=3):
    """Read every sheet of an Excel file, detecting the header row. Returns {sheet: DataFrame}."""
    try:
        import openpyxl  # noqa: F401   (needed to read .xlsx files)
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openpyxl"])
    xl = pd.ExcelFile(path)
    print("Sheets in", os.path.basename(path), ":", xl.sheet_names)
    out = {}
    for sheet in xl.sheet_names:
        raw = xl.parse(sheet, header=None, dtype=str)
        raw = raw.dropna(how="all").dropna(axis=1, how="all").reset_index(drop=True)
        if len(raw) < min_rows + 1:
            continue
        h = next((i for i in range(min(40,len(raw))) if any(
            snake(v) in ('codgeo','com','code','code_insee','depcom')
            or snake(v).startswith('codgeo_') for v in raw.iloc[i].dropna())), None)
        if h is None:
            continue
        df = raw.iloc[h + 1:].copy()
        df.columns = [str(c) if pd.notna(c) else f"col_{i}" for i, c in enumerate(raw.iloc[h])]
        out[sheet] = clean_cols(df.reset_index(drop=True))
    return out

def insee_links(page_id):
    """Find the download links published on an INSEE statistics page."""
    try:
        html = session.get(f"https://www.insee.fr/fr/statistiques/{page_id}", timeout=60).text
    except Exception as e:
        print("INSEE page unreachable:", page_id, str(e)[:100]); return []
    found = re.findall(rf"/fr/statistiques/fichier/{page_id}/[^\"'<>\s]+", html)
    return ["https://www.insee.fr" + u.replace("&amp;", "&") for u in dict.fromkeys(found)]

def datagouv_resources(dataset):
    r = session.get(f"https://www.data.gouv.fr/api/1/datasets/{dataset}/", timeout=60).json()
    return r.get("resources", [])
def load_json(path):
    with open(path,encoding='utf-8') as fh: return json.load(fh)

def to_delta(df,name):
    if df is None or df.empty: return
    write_table(BRONZE,name,clean_cols(df.copy()))

def required_download(url,source,filename):
    p=download(url,source,filename,skip_existing=True)
    if not p: raise RuntimeError(f"Téléchargement obligatoire échoué: {source}")
    return p


In [ ]:
# Communes: l'API interdit les contours sans filtre. Parcours par département,
# y compris lorsque DEPARTEMENTS=[] (France entière disponible dans le catalogue).
def fetch_geo_json(url, params=None, attempts=3):
    last_error = None
    for attempt in range(attempts):
        try:
            response = session.get(url, params=params, timeout=(15, 180))
            response.raise_for_status()
            payload = response.json()
            if not isinstance(payload, list):
                raise ValueError('Réponse géographique inattendue: liste attendue')
            return payload
        except (requests.RequestException, ValueError) as exc:
            last_error = exc
            # Une erreur de requête 4xx (hors limitation) ne sera pas résolue par un nouvel essai.
            resp = getattr(exc, 'response', None)
            if resp is not None and 400 <= resp.status_code < 500 and resp.status_code != 429:
                break
            if attempt + 1 < attempts:
                time.sleep(2 ** attempt)
    raise RuntimeError(f"API géographique inaccessible après essais: {url}. Détail: {last_error}") from last_error

if DEPARTEMENTS:
    selected_departments = sorted({str(dep).strip().upper().zfill(2) for dep in DEPARTEMENTS})
else:
    department_catalogue = fetch_geo_json('https://geo.api.gouv.fr/departements', {'fields': 'code'})
    selected_departments = sorted({str(item['code']) for item in department_catalogue})
    if not selected_departments:
        raise ValueError('Catalogue des départements vide; aucune couverture nationale ne peut être établie')

parts = []
for index, dep in enumerate(selected_departments, start=1):
    print(f"Communes et contours: département {dep} ({index}/{len(selected_departments)})", flush=True)
    url = f"https://geo.api.gouv.fr/departements/{dep}/communes"
    data = fetch_geo_json(url, {
        'fields': 'code,nom,centre,contour,population,codeDepartement,codeRegion,surface',
        'format': 'json',
        'geometry': 'contour',
    })
    if not data:
        raise ValueError(f'Aucune commune reçue pour le département {dep}; collecte interrompue')
    with open(out_path('communes', f'{dep}.json'), 'w', encoding='utf-8') as fh:
        json.dump(data, fh, ensure_ascii=False)
    for item in data:
        centre = (item.get('centre') or {}).get('coordinates', [None, None])
        parts.append({
            'code': item['code'], 'nom': item['nom'],
            'code_dept': item.get('codeDepartement'), 'code_region': item.get('codeRegion'),
            'population_api': item.get('population'), 'surface_ha': item.get('surface'),
            'lon': centre[0], 'lat': centre[1], 'contour_json': json.dumps(item.get('contour')),
        })
communes = pd.DataFrame(parts)
if communes.empty:
    raise ValueError('Aucune commune dans le périmètre')
assert_unique(communes, ['code'], 'communes')
to_delta(communes, 'communes')


In [ ]:
# Catalogue national + métadonnées des ressources; les erreurs de flux sont tracées.
r=session.get("https://transport.data.gouv.fr/api/datasets",timeout=90); r.raise_for_status()
catalogue=r.json()
if not isinstance(catalogue,list): raise ValueError("Structure catalogue PAN inattendue")
with open(out_path('transport_catalogue','datasets.json'),'w') as f: json.dump(catalogue,f)
rows=[]
for d in catalogue:
    for r in d.get('resources',[]):
        rows.append({'dataset_id':str(d.get('id') or d.get('datagouv_id')),'slug':d.get('slug'),
          'title':d.get('title'),'type':d.get('type'),'resource_id':str(r.get('id')),
          'format':str(r.get('format','')).lower(),'url':r.get('url')})
cat=pd.DataFrame(rows); to_delta(cat,'transport_catalogue')
def selected(d,ids):
    return not ids or str(d.get('id') or d.get('datagouv_id')) in ids or d.get('slug') in ids
gtfs=[(d,r) for d in catalogue if selected(d,GTFS_DATASET_IDS)
      for r in d.get('resources',[]) if str(r.get('format','')).lower()=='gtfs']
gbfs=[(d,r) for d in catalogue if selected(d,GBFS_DATASET_IDS)
      for r in d.get('resources',[]) if str(r.get('format','')).lower()=='gbfs']
# Un GTFS par URL. Les doublons géographiques restent identifiables par leur réseau.
gtfs=list({r['url']:(d,r) for d,r in gtfs if r.get('url')}.values())
stop_parts=[]; feed_log=[]
for d,r in gtfs[:MAX_GTFS]:
    fid=str(r.get('id') or d.get('id')); p=download(r['url'],'gtfs',safe_name(fid)+'.zip')
    status='error'; count=0
    try:
        if not p: raise ValueError('Téléchargement échoué')
        with zipfile.ZipFile(p) as z:
            members={os.path.basename(n).lower():n for n in z.namelist()}
            with z.open(members['stops.txt']) as fh:
                t=pd.read_csv(fh,dtype=str,encoding='utf-8-sig')
            needed={'stop_id','stop_name','stop_lat','stop_lon'}
            if not needed<=set(t.columns): raise ValueError('Colonnes stops.txt manquantes')
            t=t.reindex(columns=list(needed)+['location_type','parent_station'])
            t['feed_id']=fid; t['dataset_title']=d.get('title'); stop_parts.append(t)
            count=len(t); status='ok'
    except Exception as e: print('GTFS',fid,str(e)[:160])
    feed_log.append({'feed_id':fid,'kind':'gtfs','url':r['url'],'status':status,'rows':count})
if stop_parts: to_delta(pd.concat(stop_parts,ignore_index=True),'gtfs_stops')
# GBFS: seuls les systèmes explicitement automobiles seront comptés en Silver.
def gbfs_name(v):
    if isinstance(v,str): return v
    if isinstance(v,list):
        vals=[a for a in v if isinstance(a,dict)]
        best=next((a for a in vals if a.get('language')=='fr'),vals[0] if vals else {})
        return best.get('text') or best.get('value')
    if isinstance(v,dict): return v.get('fr') or v.get('en')
    return None

def feeds_from_discovery(doc):
    data=doc.get('data',{})
    if isinstance(data.get('feeds'),list): return data['feeds']  # GBFS v3
    for lang in ['fr','en']+list(data):
        if isinstance(data.get(lang),dict) and isinstance(data[lang].get('feeds'),list):
            return data[lang]['feeds']  # GBFS v2
    raise ValueError('Découverte GBFS sans feeds (supply URL gbfs.json)')
station_parts=[]
for d,r in list({r['url']:(d,r) for d,r in gbfs if r.get('url')}.values())[:MAX_GBFS]:
    fid=str(r.get('id') or d.get('id')); status='error'; count=0
    try:
        p=download(r['url'],'gbfs',safe_name(fid)+'__gbfs.json')
        if not p: raise ValueError('Découverte échouée')
        feeds={f['name']:f['url'] for f in feeds_from_discovery(load_json(p))}
        payload={}
        for name in ['station_information','vehicle_types','system_information']:
            if name in feeds:
                p=download(feeds[name],'gbfs',safe_name(fid)+'__'+name+'.json')
                if not p: raise ValueError(f'Flux {name} échoué')
                payload[name]=load_json(p).get('data',{})
        types=payload.get('vehicle_types',{}).get('vehicle_types',[])
        car_ids=[str(t['vehicle_type_id']) for t in types if t.get('form_factor')=='car']
        title=str(d.get('title',''))+' '+str(d.get('slug',''))
        is_citiz='citiz' in title.lower()
        for s in payload.get('station_information',{}).get('stations',[]):
            row={'feed_id':fid,'station_id':str(s['station_id']),'name':gbfs_name(s.get('name')),
                 'lat':s.get('lat'),'lon':s.get('lon'),'capacity':s.get('capacity'),
                 'operator':gbfs_name(payload.get('system_information',{}).get('name')) or d.get('title'),
                 'is_citiz':is_citiz,'car_type_ids':json.dumps(car_ids),
                 'all_type_ids':json.dumps([str(t.get('vehicle_type_id')) for t in types]),
                 'station_type_ids':json.dumps(s.get('vehicle_type_ids',[])),
                 'snapshot_date':TODAY,'source_url':r['url']}
            station_parts.append(row); count+=1
        status='ok'
    except Exception as e: print('GBFS',fid,str(e)[:160])
    feed_log.append({'feed_id':fid,'kind':'gbfs','url':r['url'],'status':status,'rows':count})
if station_parts: to_delta(pd.DataFrame(station_parts),'gbfs_stations')
if feed_log: to_delta(pd.DataFrame(feed_log),'transport_feed_status')


In [ ]:
# 6. Population history (INSEE 1876-2023) -> table population_historique
p = download(HISTORIQUE_URL, "insee_population_historique", os.path.basename(HISTORIQUE_URL),
             skip_existing=True)
if p:
    try:
        sheets = excel_tables(p, min_rows=1000)        # data sheet(s) only, notes are skipped
        for sheet, df in sheets.items():
            to_delta(auto_numeric(df), f"population_historique_{snake(sheet)[:30]}")
    except Exception as e:
        print("population_historique skipped:", str(e)[:150])


In [ ]:
# 11. Housing and cars (INSEE 2022) -> table logement_2022
frames = []
for url, src in ((LOGEMENT_URL, "france_hors_mayotte"),):
    p = download(url, "insee_logement_2022", os.path.basename(url), skip_existing=True)
    if p:
        try:
            for name, df in zip_csvs(p):
                df = clean_cols(df)
                df.insert(0, "perimetre", src)
                frames.append(df)
        except Exception as e:
            print("logement unreadable:", url, str(e)[:100])
if frames:
    df = auto_numeric(pd.concat(frames, ignore_index=True))
    if {"p22_rp", "p22_rp_voit1", "p22_rp_voit2p"} <= set(df.columns):
        # Lower bound: the 2-cars-or-more group is counted as exactly two cars.
        df["voitures_par_menage_min"] = (df.p22_rp_voit1 + 2 * df.p22_rp_voit2p) / df.p22_rp
    to_delta(df, "logement_2022")


In [ ]:
# 10. Home-work flows (INSEE 2022) -> flux_domicile_travail_2022 + distance tables
link = FLUX_URL
if not link:
    cands = [u for u in insee_links(FLUX_PAGE) if "csv" in u.lower()]
    link = cands[0] if cands else ""
    print("Flow file candidates:", cands)
flux = None
if link:
    p = download(link, "insee_flux_domicile_travail", os.path.basename(link), skip_existing=True)
    try:
        frames = [clean_cols(df) for _, df in zip_csvs(p)] if p.lower().endswith(".zip") \
            else [clean_cols(read_csv_any(p))]
        flux = auto_numeric(pd.concat(frames, ignore_index=True))
        to_delta(flux, "flux_domicile_travail_2022")
    except Exception as e:
        print("flux skipped:", str(e)[:150])
else:
    print("Flows: no link found, set FLUX_URL from", f"https://www.insee.fr/fr/statistiques/{FLUX_PAGE}")


In [ ]:
# 13. Crime statistics: theft, vandalism (SSMSI) -> delinquance_communale / departementale / regionale
try:
    pref = {"parquet": 0, "csv.gz": 1, "csv": 2}
    best = {}
    for res in datagouv_resources(SSMSI_DATASET):
        title = (res.get("title") or "").lower()
        fmt = (res.get("format") or "").lower()
        if fmt not in pref:
            continue
        level = ("communale" if "commun" in title else "departementale" if "partement" in title
                 else "regionale" if "gion" in title else None)
        print("SSMSI resource:", res.get("title"), "|", fmt, "|", level)
        if level and (level not in best or pref[fmt] < pref[best[level][0]]):
            best[level] = (fmt, res)
    for level, (fmt, res) in best.items():
        p = download(res["url"], "ssmsi_delinquance", f"{level}.{fmt}", skip_existing=True)
        if not p:
            continue
        df = pd.read_parquet(p) if fmt == "parquet" else read_csv_any(p)
        for c in df.columns:
            if pd.api.types.is_object_dtype(df[c]):
                df[c] = df[c].astype("string")
        to_delta(auto_numeric(clean_cols(df)), f"delinquance_{level}")
    if not best:
        print("SSMSI: resource titles did not mention commune / departement / region. "
              "Use EXTRA_DATAGOUV to load them generically.")
except Exception as e:
    print("delinquance skipped:", str(e)[:150])


In [ ]:
import sys
import subprocess
import importlib

try:
    import pyproj
except ModuleNotFoundError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "pyproj"
    ])
    importlib.invalidate_caches()
    import pyproj
# Carreaux INSEE: lecture par morceaux, projection explicite selon l'identifiant.
# On conserve le millésime ET l'indicatrice d'imputation; aucun prorata communal inventé.
if DOWNLOAD_GRID:
    from pyproj import Transformer
    p=download(FILOSOFI_URL,'insee_filosofi_200m',os.path.basename(FILOSOFI_URL),skip_existing=True)
    frames=[]
    if p:
        with zipfile.ZipFile(p) as z:
            for member in z.namelist():
                if not member.lower().endswith('.csv'): continue
                with z.open(member) as f: sep=sniff_sep(f.read(8192))
                with z.open(member) as f:
                    for chunk in pd.read_csv(f,sep=sep,dtype=str,chunksize=CSV_CHUNKSIZE):
                        t=clean_cols(chunk)
                        ic=next((c for c in ['idcar_200m','idcar_200','idcar'] if c in t),None)
                        pc=next((c for c in ['ind','individus','population'] if c in t),None)
                        if not ic or not pc: raise ValueError(f'Carroyage: colonnes inattendues {list(t.columns)}')
                        m=t[ic].str.extract(r'(?:CRS(\d+)RES)?200mN(\d+)E(\d+)')
                        crs=nnum(m[0].fillna('3035'))
                        x=nnum(m[2])+100; y=nnum(m[1])+100
                        t['lon']=np.nan; t['lat']=np.nan
                        for epsg in crs.dropna().unique():
                            mask=(crs==epsg)&x.notna()&y.notna()
                            a,b=Transformer.from_crs(int(epsg),4326,always_xy=True).transform(x[mask],y[mask])
                            t.loc[mask,'lon']=a; t.loc[mask,'lat']=b
                        o=pd.DataFrame({'carreau_id':t[ic],'population':nnum(t[pc]),'lon':t.lon,'lat':t.lat,
                          'population_imputee':nnum(t['i_est_200']) if 'i_est_200' in t else np.nan,'annee':2021})
                        # Bbox conservatrice pour lecture; l'affectation exacte vient en Silver.
                        if DEPARTEMENTS:
                            from shapely.geometry import shape
                            bounds=[shape(json.loads(c)).bounds for c in communes.contour_json if c!='null']
                            inside=np.zeros(len(o),dtype=bool)
                            for xmin,ymin,xmax,ymax in bounds:
                                inside|=o.lon.between(xmin-.02,xmax+.02)&o.lat.between(ymin-.02,ymax+.02)
                            o=o[inside]
                        o=o[o.population>0]
                        if len(o): frames.append(o)
        if frames: to_delta(pd.concat(frames,ignore_index=True),'carreaux_population')


In [ ]:
# 14. Projections de population Omphale (INSEE 2021-2070, par département, 11 scénarios) -> projections_omphale_dep
# REPRENDRE_RUN=True : ajoute cette table au dernier run Bronze complet, sans relancer le reste
# (Silver ne lit qu'un seul run). Dans ce cas, exécuter les cellules 1 à 3 puis celle-ci, et pas la cellule de commit.
REPRENDRE_RUN = False
if REPRENDRE_RUN:
    RUN_ID = read_table(BRONZE,'pipeline_run',required=True).iloc[0]['run_id']
p = download(OMPHALE_URL, "insee_projections_omphale", os.path.basename(OMPHALE_URL), skip_existing=True)
if p:
    frames = []
    with zipfile.ZipFile(p) as z:
        for name in sorted(n for n in z.namelist() if n.endswith(".xlsx")):
            # Population en milliers; l'en-tête DEP, LIBDEP, POP_2018 ... POP_2070 est en 6e ligne.
            df = pd.read_excel(z.open(name), sheet_name="Population_DEP", header=5, dtype=str).dropna(subset=["DEP"])
            df.insert(0, "scenario", name.removeprefix("projections_scenario_").removesuffix(".xlsx"))
            frames.append(df)
    omphale = pd.concat(frames, ignore_index=True)
    if not {"DEP", "POP_2023", "POP_2035"} <= set(omphale.columns): raise ValueError("Schéma Omphale inattendu")
    to_delta(omphale, "projections_omphale_dep")
    print(len(omphale), "lignes Omphale, run", RUN_ID)

In [ ]:
# Contrats facultatifs stables: fichiers CSV locaux; pas de scraping de plateformes commerciales.
INPUTS = {
 'candidats':['candidat_id','commune_code','nom','lat','lon'],
 'stations_manuelles':['station_id','operator','is_citiz','lat','lon','capacity','date_observation','source'],
 'couverture_concurrence':['commune_code','inventaire_complet','date_observation','source'],
 'mobilite_mesuree':['commune_code','annee','nb_navetteurs','temps_moyen_min','source'],
 'congestion':['commune_code','date_debut','date_fin','temps_pointe_min','temps_fluide_min','nb_observations','source'],
 'marche':['candidat_id','arret_id','duree_marche_min','date_observation','source'],
 'faisabilite':['candidat_id','stationnement_valide','cout_annuel_eur','source'],
 'sinistres_citiz':['commune_code','annee','vehicules_jours','nb_vols','nb_degradations','source'],
}
for name,cols in INPUTS.items():
    p=f'{INPUT_DIR}/{name}.csv'
    if not os.path.exists(p):
        print('Complément absent:',name); continue
    t=pd.read_csv(p,dtype=str)
    if not set(cols)<=set(t): raise ValueError(f'{name}: colonnes attendues {cols}')
    if len(t): to_delta(t[cols],'input_'+name)
# Commit du run Bronze: Silver ignore les anciennes tables non rafraîchies.
if not any(s['table'].startswith('population_historique_') for s in STATUS):
    raise RuntimeError('Historique population manquant: corriger la source avant Silver')
write_table(BRONZE,'ingestion_manifest',pd.DataFrame(manifest))
write_table(BRONZE,'pipeline_run',pd.DataFrame([{'run_id':RUN_ID,'status':'complete','date_utc':dt.datetime.now(dt.timezone.utc).isoformat(),
  'departements':json.dumps(DEPARTEMENTS),'gtfs_limit':str(MAX_GTFS),'gbfs_limit':str(MAX_GBFS)}]))
print('Run Bronze terminé:',RUN_ID)
